In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Training 3-Tier Hierarchical LightGBM Pipeline Layers (`models/train_hierarchical_lightgbm_layers.ipynb`)

This notebook trains the **3-Tier 4-Model Hierarchical LightGBM Triage Architecture** using **pure raw predicted probabilities (no threshold optimization)** and **majority class downsampling** for Layer 1, Layer 2, and Layer 3B:

### 3-Tier 4-Model Downsampling & Layer Feature Subsets
1. **Layer 1 LightGBM (ESI 1 Detector)** (`deploy/lightgbm_layer1_esi1_model.rds`):
   - **Features (8 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `is_dyspnea_moderate`, `is_hypertension`, `is_tachypnea`, `is_bradypnea`, `is_tachycardia_total`.
   - **Downsampling**: Majority Non-ESI 1 class downsampled to match ESI 1 count.
   - **Evaluation**: Pure un-calibrated probability threshold (`0.50`).
2. **Layer 2 LightGBM (ESI 2/3 vs ESI 4/5 Specialist)** (`deploy/rf_esi23_esi45_extreme_model.rds`):
   - **Features (18 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `is_dyspnea_moderate`, `is_bradypnea`, `is_hypotension`, `is_bradycardia_total`, `is_tachycardia_moderate`, `hr_range`, `rr_range`, `spo2_range`, `sbp_range`, `hr_last_to_min`, `rr_last_to_min`, `spo2_last_to_max`, `hr_last_to_max`, `sbp_last_to_max`, `rr_last_to_max`.
   - **Downsampling**: ESI 1 rows removed. Majority class downsampled to match minority class size.
   - **Evaluation**: Pure probability threshold (`0.50`).
3. **Layer 3A LightGBM (ESI 2 vs ESI 3 Specialist)** (`deploy/lightgbm_esi23_model.rds`):
   - **Features (9 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `hr_mean_to_last`, `is_bradypnea`, `is_hypotension`, `is_bradycardia_total`, `spo2_mean_to_last`, `rr_mean_to_last`.
   - **Data Split**: ESI 1, 4, 5 rows removed (natural split).
   - **Evaluation**: Pure probability threshold (`0.50`).
4. **Layer 3B LightGBM (ESI 4 vs ESI 5 Specialist)** (`deploy/lightgbm_esi45_model.rds`):
   - **Features (9 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `is_bradypnea`, `is_hypotension`, `is_bradycardia_total`, `is_tachycardia_moderate`, `is_dyspnea_total`, `hr_mean_to_last`.
   - **Downsampling**: ESI 1, 2, 3 rows removed. Majority class downsampled to match minority class size.
   - **Evaluation**: Pure probability threshold (`0.50`).

### Combined 5-Class Soft Probabilistic Model Evaluation
Combines pure raw probabilities using the joint soft product rule.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Standard Libraries & Define Helper Functions
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
# Native R Base Helpers
stratified_partition <- function(y, p, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  train_idx <- unlist(lapply(idx_list, function(indices) {
    sample(indices, size = max(1, round(length(indices) * p)))
  }))
  return(sort(train_idx))
}
fit_scaler <- function(df_train, cols) {
  means <- colMeans(df_train[, cols, drop = FALSE], na.rm = TRUE)
  sds   <- apply(df_train[, cols, drop = FALSE], 2, sd, na.rm = TRUE)
  sds[sds == 0] <- 1
  return(list(means = means, sds = sds, cols = cols))
}
apply_scaler <- function(df, scaler) {
  df_out <- df
  for (c in scaler$cols) {
    df_out[[c]] <- (df[[c]] - scaler$means[c]) / scaler$sds[c]
  }
  return(df_out)
}
downsample_binary_df <- function(df, label_vec, seed = 42) {
  set.seed(seed)
  pos_idx <- which(label_vec == 1)
  neg_idx <- which(label_vec == 0)
  n_pos <- length(pos_idx)
  n_neg <- length(neg_idx)
  if (n_pos == 0 || n_neg == 0) return(df)
  min_n <- min(n_pos, n_neg)
  sampled_pos <- if (n_pos > min_n) sample(pos_idx, min_n) else pos_idx
  sampled_neg <- if (n_neg > min_n) sample(neg_idx, min_n) else neg_idx
  return(df[sort(c(sampled_pos, sampled_neg)), ])
}
compute_cm_metrics <- function(preds, refs) {
  tbl <- table(Prediction = preds, Reference = refs)
  if (all(levels(refs) %in% c("1", "0"))) {
    tp <- ifelse("1" %in% rownames(tbl) && "1" %in% colnames(tbl), tbl["1", "1"], 0)
    fp <- ifelse("1" %in% rownames(tbl) && "0" %in% colnames(tbl), tbl["1", "0"], 0)
    fn <- ifelse("0" %in% rownames(tbl) && "1" %in% colnames(tbl), tbl["0", "1"], 0)
    tn <- ifelse("0" %in% rownames(tbl) && "0" %in% colnames(tbl), tbl["0", "0"], 0)
    sens <- ifelse((tp + fn) > 0, tp / (tp + fn), 0)
    spec <- ifelse((tn + fp) > 0, tn / (tn + fp), 0)
    bal  <- (sens + spec) / 2
    return(list(table = tbl, Sensitivity = sens, Specificity = spec, BalancedAccuracy = bal))
  } else {
    lvls <- levels(refs)
    sens_v <- numeric(length(lvls))
    spec_v <- numeric(length(lvls))
    for (i in seq_along(lvls)) {
      l <- lvls[i]
      tp <- ifelse(l %in% rownames(tbl) && l %in% colnames(tbl), tbl[l, l], 0)
      fn <- sum(tbl[, l]) - tp
      fp <- sum(tbl[l, ]) - tp
      tn <- sum(tbl) - (tp + fn + fp)
      sens_v[i] <- ifelse((tp + fn) > 0, tp / (tp + fn), 0)
      spec_v[i] <- ifelse((tn + fp) > 0, tn / (tn + fp), 0)
    }
    bal_v <- (sens_v + spec_v) / 2
    return(list(table = tbl, Sensitivity = sens_v, Specificity = spec_v, BalancedAccuracy = bal_v))
  }
}
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data & Construct Exclusive Layer Feature Sets
# ---------------------------------------------------------
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
pulse_last <- get_vec("pulse_last"); pulse_max <- get_vec("pulse_max"); pulse_min <- get_vec("pulse_min")
sbp_last   <- get_vec("sbp_last");   sbp_max   <- get_vec("sbp_max");   sbp_min   <- get_vec("sbp_min")
spo2_last  <- get_vec("spo2_last");  spo2_max  <- get_vec("spo2_max");  spo2_min  <- get_vec("spo2_min")
resp_last  <- get_vec("resp_last");  resp_max  <- get_vec("resp_max");  resp_min  <- get_vec("resp_min")
t_hr       <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_master <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(t_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(t_o2 > 90 & t_o2 < 94, 1, 0),
  is_hypertension         = ifelse(t_sbp > 220, 1, 0),
  is_hypotension          = ifelse(t_sbp <= 90, 1, 0),
  is_tachypnea            = ifelse(t_rr > 30, 1, 0),
  is_bradypnea            = ifelse(t_rr < 10, 1, 0),
  is_tachycardia_total    = ifelse(t_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(t_hr > 100 & t_hr < 150, 1, 0),
  is_bradycardia_total    = ifelse(t_hr < 40, 1, 0),
  hr_mean_to_last         = t_hr - pulse_last,
  spo2_mean_to_last       = t_o2 - spo2_last,
  rr_mean_to_last         = t_rr - resp_last,
  hr_range                = pulse_max - pulse_min,
  rr_range                = resp_max - resp_min,
  spo2_range              = spo2_max - spo2_min,
  sbp_range               = sbp_max - sbp_min,
  hr_last_to_min          = pulse_last - pulse_min,
  rr_last_to_min          = resp_last - resp_min,
  spo2_last_to_max        = spo2_last - spo2_max,
  hr_last_to_max          = pulse_last - pulse_max,
  sbp_last_to_max         = sbp_last - sbp_max,
  rr_last_to_max          = resp_last - resp_max
)
l1_feat_names  <- c("age", "gender", "cc_breathingdifficulty", "is_dyspnea_moderate", "is_hypertension", "is_tachypnea", "is_bradypnea", "is_tachycardia_total")
l2_feat_names  <- c("age", "gender", "cc_breathingdifficulty", "is_dyspnea_moderate", "is_bradypnea", "is_hypotension", "is_bradycardia_total", "is_tachycardia_moderate", "hr_range", "rr_range", "spo2_range", "sbp_range", "hr_last_to_min", "rr_last_to_min", "spo2_last_to_max", "hr_last_to_max", "sbp_last_to_max", "rr_last_to_max")
l3a_feat_names <- c("age", "gender", "cc_breathingdifficulty", "hr_mean_to_last", "is_bradypnea", "is_hypotension", "is_bradycardia_total", "spo2_mean_to_last", "rr_mean_to_last")
l3b_feat_names <- c("age", "gender", "cc_breathingdifficulty", "is_bradypnea", "is_hypotension", "is_bradycardia_total", "is_tachycardia_moderate", "is_dyspnea_total", "hr_mean_to_last")
raw_esi <- as.character(raw_df[[target_col_name]])
df_master$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_master <- na.omit(df_master)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- stratified_partition(df_master$target_col, p = 1 - test_size, seed = config$training$random_state)
train_val_df <- df_master[in_train_val, ]
test_df      <- df_master[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- stratified_partition(train_val_df$target_col, p = 1 - rel_val_size, seed = config$training$random_state)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
cont_cols <- c("age", "hr_mean_to_last", "spo2_mean_to_last", "rr_mean_to_last", "hr_range", "rr_range", "spo2_range", "sbp_range", "hr_last_to_min", "rr_last_to_min", "spo2_last_to_max", "hr_last_to_max", "sbp_last_to_max", "rr_last_to_max")
scaler <- fit_scaler(train_df, cont_cols)
train_scaled <- apply_scaler(train_df, scaler)
val_scaled   <- apply_scaler(val_df, scaler)
test_scaled  <- apply_scaler(test_df, scaler)
cat(sprintf("Partitions Prepared: Train=%d, Val=%d, Test=%d\n", nrow(train_scaled), nrow(val_scaled), nrow(test_scaled)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Train Layer 1 LightGBM (ESI 1 Detector - Majority Downsampled)
# ---------------------------------------------------------
set.seed(config$training$random_state)
y_tr_l1_full <- ifelse(train_scaled$target_col == "1", 1, 0)
y_vl_l1      <- ifelse(val_scaled$target_col == "1", 1, 0)
y_ts_l1      <- ifelse(test_scaled$target_col == "1", 1, 0)
# DOWNSAMPLE MAJORITY CLASS FOR LAYER 1
train_l1_ds <- downsample_binary_df(train_scaled, y_tr_l1_full, seed = config$training$random_state)
y_tr_l1     <- ifelse(train_l1_ds$target_col == "1", 1, 0)
dtrain_l1 <- lgb.Dataset(data = as.matrix(train_l1_ds[, l1_feat_names]), label = y_tr_l1)
dval_l1   <- lgb.Dataset(data = as.matrix(val_scaled[, l1_feat_names]),   label = y_vl_l1)
params_l1 <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l1_model <- lgb.train(
  params                = params_l1,
  data                  = dtrain_l1,
  nrounds               = 100,
  valids                = list(train = dtrain_l1, val = dval_l1),
  early_stopping_rounds = 10,
  verbose               = 0
)
# Pure raw predicted probability evaluation at 0.50 threshold
p_l1_test   <- predict(lgb_l1_model, as.matrix(test_scaled[, l1_feat_names]))
auc_l1_test <- as.numeric(pROC::roc(y_ts_l1, p_l1_test)$auc)
cm_l1 <- compute_cm_metrics(factor(ifelse(p_l1_test >= 0.50, 1, 0), levels = c(1, 0)), factor(y_ts_l1, levels = c(1, 0)))
rec_l1  <- cm_l1$Sensitivity
spec_l1 <- cm_l1$Specificity
bal_l1  <- cm_l1$BalancedAccuracy
cat("============================================================\n")
cat("   LAYER 1 LIGHTGBM (ESI 1 DETECTOR - DOWNSAMPLED, P >= 0.5) TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  ESI 1 Sensitivity (Recall) : %.4f\n", rec_l1))
cat(sprintf("  Non-ESI 1 Specificity      : %.4f\n", spec_l1))
cat(sprintf("  Balanced Accuracy          : %.4f\n", bal_l1))
cat(sprintf("  ROC-AUC (Test)             : %.4f\n", auc_l1_test))
cat("============================================================\n\n")
print(cm_l1$table)
cat("\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Layer 2 LightGBM (ESI 2/3 vs ESI 4/5 Specialist - Majority Downsampled)
# ---------------------------------------------------------
set.seed(config$training$random_state)
# REMOVE ESI 1 ROWS FOR LAYER 2
train_l2_raw <- train_scaled %>% filter(target_col != "1")
val_l2       <- val_scaled   %>% filter(target_col != "1")
test_l2      <- test_scaled  %>% filter(target_col != "1")
y_tr_l2_raw <- ifelse(train_l2_raw$target_col %in% c("2", "3"), 1, 0)
y_vl_l2     <- ifelse(val_l2$target_col %in% c("2", "3"), 1, 0)
y_ts_l2     <- ifelse(test_l2$target_col %in% c("2", "3"), 1, 0)
# DOWNSAMPLE MAJORITY CLASS FOR LAYER 2
train_l2_ds <- downsample_binary_df(train_l2_raw, y_tr_l2_raw, seed = config$training$random_state)
y_tr_l2     <- ifelse(train_l2_ds$target_col %in% c("2", "3"), 1, 0)
dtrain_l2 <- lgb.Dataset(data = as.matrix(train_l2_ds[, l2_feat_names]), label = y_tr_l2)
dval_l2   <- lgb.Dataset(data = as.matrix(val_l2[, l2_feat_names]),       label = y_vl_l2)
params_l2 <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l2_model <- lgb.train(
  params                = params_l2,
  data                  = dtrain_l2,
  nrounds               = 100,
  valids                = list(train = dtrain_l2, val = dval_l2),
  early_stopping_rounds = 10,
  verbose               = 0
)
# Pure raw probability evaluation at 0.50 threshold
p_l2_test   <- predict(lgb_l2_model, as.matrix(test_l2[, l2_feat_names]))
auc_l2_test <- as.numeric(pROC::roc(y_ts_l2, p_l2_test)$auc)
cm_l2 <- compute_cm_metrics(factor(ifelse(p_l2_test >= 0.50, 1, 0), levels = c(1, 0)), factor(y_ts_l2, levels = c(1, 0)))
rec_l2  <- cm_l2$Sensitivity
spec_l2 <- cm_l2$Specificity
bal_l2  <- cm_l2$BalancedAccuracy
cat("============================================================\n")
cat("   LAYER 2 LIGHTGBM (ESI 2/3 vs 4/5 SPECIALIST - DOWNSAMPLED, P >= 0.5) TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  ESI 2/3 Recall (Sens)      : %.4f\n", rec_l2))
cat(sprintf("  ESI 4/5 Specificity        : %.4f\n", spec_l2))
cat(sprintf("  Balanced Accuracy          : %.4f\n", bal_l2))
cat(sprintf("  ROC-AUC (Test)             : %.4f\n", auc_l2_test))
cat("============================================================\n\n")
print(cm_l2$table)
cat("\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Train Layer 3A LightGBM (ESI 2 vs ESI 3 Specialist - ESI 1,4,5 Removed)
# ---------------------------------------------------------
set.seed(config$training$random_state)
# REMOVE ESI 1, 4, 5 ROWS FOR LAYER 3A
train_l3a <- train_scaled %>% filter(target_col %in% c("2", "3"))
val_l3a   <- val_scaled   %>% filter(target_col %in% c("2", "3"))
test_l3a  <- test_scaled  %>% filter(target_col %in% c("2", "3"))
y_tr_l3a <- ifelse(train_l3a$target_col == "2", 1, 0)
y_vl_l3a <- ifelse(val_l3a$target_col == "2", 1, 0)
y_ts_l3a <- ifelse(test_l3a$target_col == "2", 1, 0)
dtrain_l3a <- lgb.Dataset(data = as.matrix(train_l3a[, l3a_feat_names]), label = y_tr_l3a)
dval_l3a   <- lgb.Dataset(data = as.matrix(val_l3a[, l3a_feat_names]),   label = y_vl_l3a)
params_l3a <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l3a_model <- lgb.train(
  params                = params_l3a,
  data                  = dtrain_l3a,
  nrounds               = 100,
  valids                = list(train = dtrain_l3a, val = dval_l3a),
  early_stopping_rounds = 10,
  verbose               = 0
)
# Pure raw probability evaluation at 0.50 threshold
p_l3a_test   <- predict(lgb_l3a_model, as.matrix(test_l3a[, l3a_feat_names]))
auc_l3a_test <- as.numeric(pROC::roc(y_ts_l3a, p_l3a_test)$auc)
cm_l3a <- compute_cm_metrics(factor(ifelse(p_l3a_test >= 0.50, 1, 0), levels = c(1, 0)), factor(y_ts_l3a, levels = c(1, 0)))
rec_l3a  <- cm_l3a$Sensitivity
spec_l3a <- cm_l3a$Specificity
bal_l3a  <- cm_l3a$BalancedAccuracy
cat("============================================================\n")
cat("   LAYER 3A LIGHTGBM (ESI 2 vs ESI 3 SPECIALIST - 9 FEATS, P >= 0.5) TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  ESI 2 Sensitivity (Recall) : %.4f\n", rec_l3a))
cat(sprintf("  ESI 3 Specificity          : %.4f\n", spec_l3a))
cat(sprintf("  Balanced Accuracy          : %.4f\n", bal_l3a))
cat(sprintf("  ROC-AUC (Test)             : %.4f\n", auc_l3a_test))
cat("============================================================\n\n")
print(cm_l3a$table)
cat("\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Train Layer 3B LightGBM (ESI 4 vs ESI 5 Specialist - Majority Downsampled)
# ---------------------------------------------------------
set.seed(config$training$random_state)
# REMOVE ESI 1, 2, 3 ROWS FOR LAYER 3B
train_l3b_raw <- train_scaled %>% filter(target_col %in% c("4", "5"))
val_l3b       <- val_scaled   %>% filter(target_col %in% c("4", "5"))
test_l3b      <- test_scaled  %>% filter(target_col %in% c("4", "5"))
y_tr_l3b_raw <- ifelse(train_l3b_raw$target_col == "4", 1, 0)
y_vl_l3b     <- ifelse(val_l3b$target_col == "4", 1, 0)
y_ts_l3b     <- ifelse(test_l3b$target_col == "4", 1, 0)
# DOWNSAMPLE MAJORITY CLASS FOR LAYER 3B
train_l3b_ds <- downsample_binary_df(train_l3b_raw, y_tr_l3b_raw, seed = config$training$random_state)
y_tr_l3b     <- ifelse(train_l3b_ds$target_col == "4", 1, 0)
dtrain_l3b <- lgb.Dataset(data = as.matrix(train_l3b_ds[, l3b_feat_names]), label = y_tr_l3b)
dval_l3b   <- lgb.Dataset(data = as.matrix(val_l3b[, l3b_feat_names]),       label = y_vl_l3b)
params_l3b <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l3b_model <- lgb.train(
  params                = params_l3b,
  data                  = dtrain_l3b,
  nrounds               = 100,
  valids                = list(train = dtrain_l3b, val = dval_l3b),
  early_stopping_rounds = 10,
  verbose               = 0
)
# Pure raw probability evaluation at 0.50 threshold
p_l3b_test   <- predict(lgb_l3b_model, as.matrix(test_l3b[, l3b_feat_names]))
auc_l3b_test <- as.numeric(pROC::roc(y_ts_l3b, p_l3b_test)$auc)
cm_l3b <- compute_cm_metrics(factor(ifelse(p_l3b_test >= 0.50, 1, 0), levels = c(1, 0)), factor(y_ts_l3b, levels = c(1, 0)))
rec_l3b  <- cm_l3b$Sensitivity
spec_l3b <- cm_l3b$Specificity
bal_l3b  <- cm_l3b$BalancedAccuracy
cat("============================================================\n")
cat("   LAYER 3B LIGHTGBM (ESI 4 vs ESI 5 SPECIALIST - DOWNSAMPLED, P >= 0.5) TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  ESI 4 Sensitivity (Recall) : %.4f\n", rec_l3b))
cat(sprintf("  ESI 5 Specificity          : %.4f\n", spec_l3b))
cat(sprintf("  Balanced Accuracy          : %.4f\n", bal_l3b))
cat(sprintf("  ROC-AUC (Test)             : %.4f\n", auc_l3b_test))
cat("============================================================\n\n")
print(cm_l3b$table)
cat("\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save All 4 Model Artifacts to deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
saveRDS(list(model = lgb_l1_model,  scaler = scaler, is_l1_lgb  = TRUE), file = file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
saveRDS(list(model = lgb_l2_model,  scaler = scaler, is_lgb_l2  = TRUE), file = file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
saveRDS(list(model = lgb_l3a_model, scaler = scaler, is_lgb_l3a = TRUE), file = file.path(deploy_dir, "lightgbm_esi23_model.rds"))
saveRDS(list(model = lgb_l3b_model, scaler = scaler, is_lgb_l3b = TRUE), file = file.path(deploy_dir, "lightgbm_esi45_model.rds"))
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
l1_l4_report <- data.frame(
  Layer = c("Layer1_ESI1_Detector", "Layer2_ESI23_vs_ESI45", "Layer3A_ESI2_vs_ESI3", "Layer3B_ESI4_vs_ESI5"),
  Input_Features = c("8_Features", "18_Features", "9_Features", "9_Features"),
  Downsampled = c("Yes", "Yes", "No", "Yes"),
  Recall = c(round(rec_l1, 4), round(rec_l2, 4), round(rec_l3a, 4), round(rec_l3b, 4)),
  Specificity = c(round(spec_l1, 4), round(spec_l2, 4), round(spec_l3a, 4), round(spec_l3b, 4)),
  Balanced_Accuracy = c(round(bal_l1, 4), round(bal_l2, 4), round(bal_l3a, 4), round(bal_l3b, 4)),
  ROC_AUC = c(round(auc_l1_test, 4), round(auc_l2_test, 4), round(auc_l3a_test, 4), round(auc_l3b_test, 4))
)
write.csv(l1_l4_report, file = file.path(reports_dir, "hierarchical_l1_l2_training_report.csv"), row.names = FALSE)
cat("All 4 Sub-Model Artifacts saved to deploy/ & Report saved to reports/hierarchical_l1_l2_training_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 8: Evaluate Combined 5-Class Soft Probabilistic Pipeline (Pure Raw Probabilities)
# ---------------------------------------------------------
X_test_l1  <- as.matrix(test_scaled[, l1_feat_names])
X_test_l2  <- as.matrix(test_scaled[, l2_feat_names])
X_test_l3a <- as.matrix(test_scaled[, l3a_feat_names])
X_test_l3b <- as.matrix(test_scaled[, l3b_feat_names])
p1_test  <- predict(lgb_l1_model,  X_test_l1)
p2_test  <- predict(lgb_l2_model,  X_test_l2)
p3a_test <- predict(lgb_l3a_model, X_test_l3a)
p3b_test <- predict(lgb_l3b_model, X_test_l3b)
probs_comb <- matrix(0, nrow = nrow(test_df), ncol = 5)
colnames(probs_comb) <- c("1", "2", "3", "4", "5")
# Joint Soft Probability Product using Pure Raw Model Probabilities
probs_comb[, 1] <- p1_test
probs_comb[, 2] <- (1 - p1_test) * p2_test * p3a_test
probs_comb[, 3] <- (1 - p1_test) * p2_test * (1 - p3a_test)
probs_comb[, 4] <- (1 - p1_test) * (1 - p2_test) * p3b_test
probs_comb[, 5] <- (1 - p1_test) * (1 - p2_test) * (1 - p3b_test)
preds_comb <- factor(apply(probs_comb, 1, which.max), levels = 1:5)
act_comb   <- factor(as.numeric(as.character(test_df$target_col)), levels = 1:5)
cm_comb <- compute_cm_metrics(preds_comb, act_comb)
rec_comb_list  <- cm_comb$Sensitivity
spec_comb_list <- cm_comb$Specificity
bal_comb_list  <- cm_comb$BalancedAccuracy
auc_comb_list <- sapply(1:5, function(i) {
  act_bin <- ifelse(act_comb == i, 1, 0)
  r_obj <- tryCatch(pROC::roc(act_bin, probs_comb[, i]), error = function(e) NULL)
  if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
})
macro_rec_comb  <- mean(rec_comb_list)
macro_spec_comb <- mean(spec_comb_list)
macro_bal_comb  <- mean(bal_comb_list)
macro_auc_comb  <- mean(auc_comb_list, na.rm = TRUE)
cat("============================================================\n")
cat("   COMBINED 5-CLASS SOFT PROBABILISTIC MODEL TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  Macro Recall (Sensitivity) : %.4f\n", macro_rec_comb))
cat(sprintf("  Macro Specificity          : %.4f\n", macro_spec_comb))
cat(sprintf("  Macro Balanced Accuracy    : %.4f\n", macro_bal_comb))
cat(sprintf("  Macro ROC-AUC              : %.4f\n", macro_auc_comb))
cat("============================================================\n\n")
cat("Confusion Matrix (Combined 5-Class Soft Probabilistic Model):\n")
print(cm_comb$table)
cat("\n\n")
soft_comb_report <- data.frame(
  Pipeline = "Combined_5Class_Soft_Probability_Model",
  Macro_Recall = round(macro_rec_comb, 4),
  Macro_Specificity = round(macro_spec_comb, 4),
  Macro_Balanced_Accuracy = round(macro_bal_comb, 4),
  Macro_ROC_AUC = round(macro_auc_comb, 4)
)
write.csv(soft_comb_report, file = file.path(reports_dir, "combined_soft_pipeline_training_report.csv"), row.names = FALSE)
cat("Combined Soft Probability Model Training Report saved to reports/combined_soft_pipeline_training_report.csv\n")